In [1]:
# This is for entire data preprocessing.

import torch
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import os
import glob
from tqdm import tqdm
import json

# --- 1. GPU Configuration ---
if torch.cuda.is_available():
    print(f"Using GPU: {torch.cuda.get_device_name()}")
else:
    device = torch.device("cpu")
    print("CUDA not available. Using CPU.")

# --- 2. Load and Combine Dataset ---
# This section remains unchanged. It loads all raw CSVs into a single DataFrame.
data_path = 'data/'
all_files = glob.glob(os.path.join(data_path, "*.csv"))

df_list = []
for filename in tqdm(all_files, desc="Loading CSVs"):
    df_list.append(pd.read_csv(filename))

df = pd.concat(df_list, ignore_index=True)

print(f"Initial dataset shape: {df.shape}")


# --- 3. Data Cleaning and Preprocessing for Binary Classification ---
# Clean column names (remove leading/trailing spaces)
df.columns = df.columns.str.strip()

# Handle infinite values by replacing them with NaN, then drop all rows with NaN values.
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)

print(f"Shape after dropping NaNs/Infs: {df.shape}")

# Separate features (X) and the original labels (y)
X = df.drop('Label', axis=1)
y = df['Label']

# --- MODIFICATION FOR BINARY CLASSIFICATION ---
# Instead of mapping each unique label to a number, we will map them to two classes.
# 1 -> Malicious (any label that is NOT 'BENIGN')
# 0 -> BENIGN
print("Converting to binary classification: 1 for Malicious, 0 for BENIGN.")
y_binary = (y != 'BENIGN').astype(int)

# Define the new, simpler label mapping for reference.
binary_label_mapping = {'BENIGN': 0, 'Malicious': 1}
n_classes = len(binary_label_mapping)
print(f"Found {n_classes} unique classes after binary conversion.")
print(binary_label_mapping)


# --- 4. Balancing the Dataset ---
# This logic remains valid. We downsample the 'BENIGN' class to be roughly
# equal to the total number of malicious samples.

# Count the number of malicious samples (where the label is 1).
attack_count = y_binary[y_binary == 1].count()
# Get the indices of all the benign samples.
benign_indices = df[df['Label'] == 'BENIGN'].index

# Randomly select a subset of benign indices that matches the attack count.
random_indices = np.random.choice(benign_indices, attack_count, replace=False)

# Get the indices of all attack samples.
attack_indices = df[df['Label'] != 'BENIGN'].index

# Concatenate the attack indices and the downsampled benign indices.
under_sample_indices = np.concatenate([attack_indices, random_indices])
df_balanced = df.loc[under_sample_indices]

# Re-create the balanced features and labels from the balanced DataFrame.
X_balanced = df_balanced.drop('Label', axis=1)

# --- MODIFICATION FOR BINARY CLASSIFICATION ---
# Apply the same binary mapping to the 'Label' column of the now-balanced DataFrame.
y_balanced_binary = (df_balanced['Label'] != 'BENIGN').astype(int)

print(f"Shape after balancing: {X_balanced.shape}")
print(f"Value counts in balanced labels:\n{y_balanced_binary.value_counts()}")


# --- 5. Feature Scaling ---
# This section remains unchanged. We apply Z-score normalization to the features.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_balanced)


# --- 6. Train-Test Split ---
# This section remains unchanged. We split into 80% training and 20% testing.
# Stratifying by the binary label ensures both sets have a similar proportion
# of malicious vs. benign samples.
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_balanced_binary.values, test_size=0.2, random_state=42, stratify=y_balanced_binary
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")


# --- 7. Save Processed Data ---
# Save the processed numpy arrays to avoid re-running this expensive step.
processed_data_path = 'processed_data/' # Using a new folder for clarity
if not os.path.exists(processed_data_path):
    os.makedirs(processed_data_path)

np.save(os.path.join(processed_data_path, 'X_train.npy'), X_train)
np.save(os.path.join(processed_data_path, 'X_test.npy'), X_test)
np.save(os.path.join(processed_data_path, 'y_train.npy'), y_train)
np.save(os.path.join(processed_data_path, 'y_test.npy'), y_test)

# --- MODIFICATION FOR BINARY CLASSIFICATION ---
# Save the new binary label mapping for later use.
with open(os.path.join(processed_data_path, 'label_mapping.json'), 'w') as f:
    json.dump(binary_label_mapping, f)

print("Preprocessing for binary classification complete. Processed data saved.")

Using GPU: NVIDIA RTX 4000 Ada Generation


Loading CSVs: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 8/8 [00:07<00:00,  1.07it/s]


Initial dataset shape: (2830743, 79)
Shape after dropping NaNs/Infs: (2827876, 79)
Converting to binary classification: 1 for Malicious, 0 for BENIGN.
Found 2 unique classes after binary conversion.
{'BENIGN': 0, 'Malicious': 1}
Shape after balancing: (1113112, 78)
Value counts in balanced labels:
Label
1    556556
0    556556
Name: count, dtype: int64
X_train shape: (890489, 78)
X_test shape: (222623, 78)
y_train shape: (890489,)
y_test shape: (222623,)
Preprocessing for binary classification complete. Processed data saved.


In [2]:
import torch
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import os
import glob
from tqdm import tqdm
import json

# ==============================================================================
# 1. EXPERIMENT CONFIGURATION
# ==============================================================================
ATTACK_MODIFICATION_RULES = {
    "DoS Hulk":       0.9782697,   # removed 225,124 / 230,124
    "PortScan":       0.9685132,   # removed 153,804 / 158,804
    "DDoS":           0.9609451,   # removed 123,025 / 128,025
    "DoS GoldenEye":  0.5142184,   # removed   5,293 / 10,293
    "FTP-Patator":    0.3698803    # removed   2,935 /  7,935
}

# ==============================================================================
# 2. GPU Configuration and Data Loading
# ==============================================================================
if torch.cuda.is_available():
    print(f"Using GPU: {torch.cuda.get_device_name()}")
else:
    print("CUDA not available. Using CPU.")

data_path = 'data/'
all_files = glob.glob(os.path.join(data_path, "*.csv"))
df_list = []
for filename in tqdm(all_files, desc="Loading CSVs"):
    df_list.append(pd.read_csv(filename))
df = pd.concat(df_list, ignore_index=True)
print(f"\nInitial dataset shape: {df.shape}")

# ==============================================================================
# 3. Data Cleaning and Initial Analysis
# ==============================================================================
df.columns = df.columns.str.strip()
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)
print(f"Shape after dropping NaNs/Infs: {df.shape}")

print("\n--- Analyzing Attack Distribution in the Original Dataset ---")
attack_labels_df = df[df['Label'] != 'BENIGN']
attack_counts = attack_labels_df['Label'].value_counts()
print("Total counts for each attack type:")
print(attack_counts)
print("----------------------------------------------------------\n")
print(f"EXPERIMENTAL SETUP: Applying the following modification rules:")
for attack, ratio in ATTACK_MODIFICATION_RULES.items():
    print(f" -> '{attack}': Remove {ratio*100:.0f}% from training pool")
print(" -> All other attacks will be fully included in the training pool.")

# ==============================================================================
# 4. Data Segregation based on Dictionary Rules
# ==============================================================================
print("\n--- Segregating Data based on Dictionary Rules ---")

all_attack_types = attack_labels_df['Label'].unique()
main_pool_malicious_list = []
holdout_pool_list = []

for attack_type in all_attack_types:
    ratio_to_remove = ATTACK_MODIFICATION_RULES.get(attack_type, 0.0)
    df_current_attack = df[df['Label'] == attack_type]

    if ratio_to_remove > 0.0:
        train_ratio_to_keep = 1.0 - ratio_to_remove
        if train_ratio_to_keep > 0:
            df_keep, df_holdout = train_test_split(df_current_attack, train_size=train_ratio_to_keep, random_state=42)
        else: # Handle ratio_to_remove = 1.0
            df_keep, df_holdout = pd.DataFrame(), df_current_attack
        
        if not df_keep.empty: main_pool_malicious_list.append(df_keep)
        if not df_holdout.empty: holdout_pool_list.append(df_holdout)
        print(f"  - '{attack_type}': Keeping {len(df_keep)} samples for training, holding out {len(df_holdout)}.")
    else:
        main_pool_malicious_list.append(df_current_attack)
        print(f"  - '{attack_type}': Keeping all {len(df_current_attack)} samples for training.")

df_main_pool_malicious = pd.concat(main_pool_malicious_list, ignore_index=True)
df_holdout = pd.concat(holdout_pool_list, ignore_index=True) if holdout_pool_list else pd.DataFrame()

# ==============================================================================
# 5. Balancing the Main Data Pool for Training
# ==============================================================================
print("\n--- Balancing the Main Training Data Pool ---")
df_benign = df[df['Label'] == 'BENIGN']
attack_count_main = len(df_main_pool_malicious)

# This DataFrame contains all the benign samples that will be used for training and initial testing
df_benign_balanced = df_benign.sample(n=attack_count_main, random_state=42)

df_balanced = pd.concat([df_main_pool_malicious, df_benign_balanced], ignore_index=True)
print(f"Shape after balancing the main data pool: {df_balanced.shape}")

X_balanced = df_balanced.drop('Label', axis=1)
y_balanced_binary = (df_balanced['Label'] != 'BENIGN').astype(int)
print(f"Value counts in the balanced pool:\n{y_balanced_binary.value_counts()}")

# ==============================================================================
# 6. Feature Scaling, Splitting, and Augmenting
# ==============================================================================
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_balanced)

X_train, X_test_initial, y_train, y_test_initial = train_test_split(
    X_scaled, y_balanced_binary.values, test_size=0.2, random_state=42, stratify=y_balanced_binary
)
print(f"\nCreated training set with shape: {X_train.shape}")
print(f"Initial test set shape (before augmenting): {X_test_initial.shape}")

if not df_holdout.empty:
    X_holdout = df_holdout.drop('Label', axis=1)
    X_holdout_scaled = scaler.transform(X_holdout)
    y_holdout = np.ones(len(X_holdout_scaled), dtype=int)
    X_test_imbalanced = np.concatenate([X_test_initial, X_holdout_scaled], axis=0)
    y_test_imbalanced = np.concatenate([y_test_initial, y_holdout], axis=0)
    print(f"Augmented test set with {len(df_holdout)} holdout samples. Imbalanced test set shape: {X_test_imbalanced.shape}")
else:
    X_test_imbalanced, y_test_imbalanced = X_test_initial, y_test_initial
    print("No holdout samples to add; test set is not augmented.")

# ==============================================================================
# 7. NEW LOGIC: Force a Perfect 50/50 Balance on the Final Test Set via Upsampling
# ==============================================================================
print("\n--- Balancing the final test set to a perfect 50/50 distribution ---")
print(f"Imbalanced test set distribution:\n{pd.Series(y_test_imbalanced).value_counts()}")

benign_count_test = np.sum(y_test_imbalanced == 0)
malicious_count_test = np.sum(y_test_imbalanced == 1)

if malicious_count_test > benign_count_test:
    print("Malicious class is the majority. Upsampling BENIGN class.")
    needed_benign = malicious_count_test - benign_count_test

    # Identify benign samples already used in training/initial testing to avoid data leakage
    used_benign_indices = df_benign_balanced.index
    # Get all benign indices from the original, full dataset
    all_benign_indices = df[df['Label'] == 'BENIGN'].index
    # The pool of available samples is the set difference
    available_benign_indices = all_benign_indices.difference(used_benign_indices)
    
    if len(available_benign_indices) < needed_benign:
        raise ValueError(f"Not enough unique BENIGN samples to upsample the test set! Needed {needed_benign}, but only {len(available_benign_indices)} are available.")

    # Randomly select the required number of fresh benign samples
    upsample_indices = np.random.choice(available_benign_indices, needed_benign, replace=False)
    df_upsample_benign = df.loc[upsample_indices]

    # IMPORTANT: Scale these new features with the SAME scaler fit on the training data
    X_upsample = df_upsample_benign.drop('Label', axis=1)
    X_upsample_scaled = scaler.transform(X_upsample)
    y_upsample = np.zeros(len(X_upsample_scaled), dtype=int)

    # Combine the imbalanced set with the new benign samples
    X_test = np.concatenate([X_test_imbalanced, X_upsample_scaled], axis=0)
    y_test = np.concatenate([y_test_imbalanced, y_upsample], axis=0)
    print(f"Upsampled test set with {needed_benign} new BENIGN samples.")

else:
    print("BENIGN class is the majority or equal. Downsampling malicious class.")
    min_count = malicious_count_test
    benign_indices_test = np.where(y_test_imbalanced == 0)[0]
    malicious_indices_test = np.where(y_test_imbalanced == 1)[0]
    
    random_benign_indices = np.random.choice(benign_indices_test, min_count, replace=False)
    final_test_indices = np.concatenate([random_benign_indices, malicious_indices_test])

    X_test = X_test_imbalanced[final_test_indices]
    y_test = y_test_imbalanced[final_test_indices]

# Shuffle the final, balanced test set
shuffle_perm = np.random.permutation(len(X_test))
X_test = X_test[shuffle_perm]
y_test = y_test[shuffle_perm]

print(f"Final balanced test set distribution:\n{pd.Series(y_test).value_counts()}")

# ==============================================================================
# 8. Save Processed Data
# ==============================================================================
processed_data_path = 'processed_data/'
os.makedirs(processed_data_path, exist_ok=True)

experiment_name = "dict_holdout_" + "_".join(s.replace(" ", "") for s in ATTACK_MODIFICATION_RULES.keys())
print(f"\n--- Saving processed data for experiment '{experiment_name}' ---")
np.save(os.path.join(processed_data_path, 'X_train.npy'), X_train)
np.save(os.path.join(processed_data_path, 'X_test.npy'), X_test)
np.save(os.path.join(processed_data_path, 'y_train.npy'), y_train)
np.save(os.path.join(processed_data_path, 'y_test.npy'), y_test)

binary_label_mapping = {'BENIGN': 0, 'Malicious': 1}
with open(os.path.join(processed_data_path, 'label_mapping.json'), 'w') as f:
    json.dump(binary_label_mapping, f)

print(f"Final training features shape: {X_train.shape}")
print(f"Final testing features shape: {X_test.shape}")
print("\nPreprocessing for the dictionary-based experiment is complete.")


Using GPU: NVIDIA RTX 4000 Ada Generation


Loading CSVs: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 8/8 [00:07<00:00,  1.05it/s]



Initial dataset shape: (2830743, 79)
Shape after dropping NaNs/Infs: (2827876, 79)

--- Analyzing Attack Distribution in the Original Dataset ---
Total counts for each attack type:
Label
DoS Hulk                      230124
PortScan                      158804
DDoS                          128025
DoS GoldenEye                  10293
FTP-Patator                     7935
SSH-Patator                     5897
DoS slowloris                   5796
DoS Slowhttptest                5499
Bot                             1956
Web Attack � Brute Force        1507
Web Attack � XSS                 652
Infiltration                      36
Web Attack � Sql Injection        21
Heartbleed                        11
Name: count, dtype: int64
----------------------------------------------------------

EXPERIMENTAL SETUP: Applying the following modification rules:
 -> 'DoS Hulk': Remove 98% from training pool
 -> 'PortScan': Remove 97% from training pool
 -> 'DDoS': Remove 96% from training pool
 -> 'DoS Go

In [3]:
import pandas as pd
import torch
import numpy as np
from sklearn.model_selection import train_test_split
import os
import json

# --- 1. Setup ---

# Configure device for PyTorch (GPU if available, otherwise CPU)
if torch.cuda.is_available():
    print(f"Using GPU: {torch.cuda.get_device_name()}")
else:
    device = torch.device("cpu")
    print("CUDA not available. Using CPU.")

# Define the file path for the dataset
file_path = 'NITK_NIDS.csv'

# Load the dataset from the CSV file into a pandas DataFrame
df = pd.read_csv(file_path)
print(f"Initial dataset shape: {df.shape}")
print(f"Initial class distribution:\n{df['Label'].value_counts()}\n")


# --- 2. Data Cleaning ---

# Remove leading/trailing whitespace from column names
df.columns = df.columns.str.strip()

# Replace infinite values with NaN (Not a Number)
df.replace([np.inf, -np.inf], np.nan, inplace=True)

# Drop any rows that contain NaN values
df.dropna(inplace=True)
print(f"Shape after dropping NaNs/Infs: {df.shape}\n")


# --- 3. Feature and Label Separation ---

# Separate the features (X) by dropping identifier and label columns
X = df.drop(['Flow ID', 'Protocol', 'Label'], axis=1)

# Isolate the target labels (y) which are still strings at this point
y = df['Label']
print(f"Shape of features (X): {X.shape}")
print(f"Shape of labels (y): {y.shape}\n")


# --- 4. Dataset Balancing (Undersampling) ---

# Count the number of malicious samples (the minority class) using the string label
attack_count = (df['Label'] == 1).sum()
print(f"Found {attack_count} malicious samples to balance against.")

# Get the indices of all benign samples
benign_indices = df[df['Label'] == 0].index

# Randomly select a subset of benign indices equal to the number of attack samples
random_indices = np.random.choice(benign_indices, attack_count, replace=False)

# Get the indices of all attack samples
attack_indices = df[df['Label'] == 1].index

# Combine the attack indices with the downsampled benign indices
under_sample_indices = np.concatenate([attack_indices, random_indices])

# Create a new balanced DataFrame using the combined indices
df_balanced = df.loc[under_sample_indices]
print(f"\nShape of the balanced DataFrame: {df_balanced.shape}")
print(f"Class distribution in balanced DataFrame:\n{df_balanced['Label'].value_counts()}\n")

# Re-create the balanced features and labels from the new DataFrame
X_balanced = df_balanced.drop(['Flow ID', 'Protocol', 'Label'], axis=1)
y_balanced_binary = df_balanced['Label']


# --- 6. Train-Test Split ---

# Split the balanced data into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(
    X_balanced, y_balanced_binary.values, test_size=0.2, random_state=42, stratify=y_balanced_binary
)

print("\n--- Shapes after Train-Test Split ---")
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")
print(f"\nClass distribution in y_train (normalized):\n{pd.Series(y_train).value_counts(normalize=True)}")
print(f"\nClass distribution in y_test (normalized):\n{pd.Series(y_test).value_counts(normalize=True)}\n")


# --- 7. Save Processed Data ---

# Define the directory to save the processed files
processed_data_path = 'processed_data/'

# Create the directory if it does not already exist
if not os.path.exists(processed_data_path):
    os.makedirs(processed_data_path)
    print(f"Created directory: {processed_data_path}")

# Save the NumPy arrays for training and testing sets
np.save(os.path.join(processed_data_path, 'X_train.npy'), X_train)
np.save(os.path.join(processed_data_path, 'X_test.npy'), X_test)
np.save(os.path.join(processed_data_path, 'y_train.npy'), y_train)
np.save(os.path.join(processed_data_path, 'y_test.npy'), y_test)


label_mapping = {'Benign': 0, 'Malicious': 1}
# Save the label mapping to a JSON file for future reference
with open(os.path.join(processed_data_path, 'label_mapping.json'), 'w') as f:
    json.dump(label_mapping, f)

# Print a confirmation message
print("Preprocessing for binary classification complete. Processed data saved.")

Using GPU: NVIDIA RTX 4000 Ada Generation
Initial dataset shape: (230873, 81)
Initial class distribution:
Label
0.0    161611
1.0     69262
Name: count, dtype: int64

Shape after dropping NaNs/Infs: (230873, 81)

Shape of features (X): (230873, 78)
Shape of labels (y): (230873,)

Found 69262 malicious samples to balance against.

Shape of the balanced DataFrame: (138524, 81)
Class distribution in balanced DataFrame:
Label
1.0    69262
0.0    69262
Name: count, dtype: int64


--- Shapes after Train-Test Split ---
X_train shape: (110819, 78)
X_test shape: (27705, 78)
y_train shape: (110819,)
y_test shape: (27705,)

Class distribution in y_train (normalized):
1.0    0.500005
0.0    0.499995
Name: proportion, dtype: float64

Class distribution in y_test (normalized):
0.0    0.500018
1.0    0.499982
Name: proportion, dtype: float64

Preprocessing for binary classification complete. Processed data saved.
